In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qdiff_theo import build_component, finite_diff
from qpu import optimize_delta, gradient_descent_finite_difference, gradient_descent_parameter_shift, build_parameterized_component
import perceval as pcvl
import pandas as pd

In [ ]:
deltas = np.logspace(-4, -1, 12)

dim = 2
i = (1, 1)
j = (1, 1)

repeats = 10
all_abs_err = []

for rep in range(repeats):
    random_theta = np.random.uniform(-np.pi, np.pi)

    decomposition = [(0, 0, random_theta)]
    target = (0, 1)

    abs_err = optimize_delta(
        dim, i, j, decomposition, target,
        random_theta, deltas, shots=2000
    )

    all_abs_err.append(abs_err)

all_abs_err = np.array(all_abs_err)

mean_abs_err = np.mean(all_abs_err, axis=0)
std_abs_err = np.std(all_abs_err, axis=0, ddof=1)

pd.DataFrame({
    "delta": deltas,
    "mean_abs_err": mean_abs_err,
    "std_abs_err": std_abs_err
}).to_csv("opt_delta_ascella.csv", index=False)

In [ ]:
df = pd.read_csv("opt_delta_ascella.csv")

plt.figure(figsize=(7, 5))

plt.errorbar(
    df["delta"],
    df["mean_abs_err"],
    yerr=df["std_abs_err"],
    fmt="o-",
    capsize=3
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel(r"$\delta$")
plt.ylabel(r"$|\Delta\partial_\theta Pr(\theta)|$")

plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("opt_delta_ascella.pdf", dpi=300)
plt.show()

In [ ]:
init_theta = 2.75

theta_opt, cost_psr = gradient_descent_parameter_shift(
    dim=2,
    i=(1,1),
    j=(1,1),
    decomposition=[
        (0,0,init_theta)
    ],
    target=(0,1),
    initial_parameter=init_theta,
    learning_rate=2.0,
    max_iter=60,
    shots=100000
)

np.save(
    "cost_psr_qpu.npy",
    cost_psr
)


In [ ]:
init_theta = 2.75

theta_opt, cost_fd = gradient_descent_finite_difference(
    dim=2,
    i=(1,1),
    j=(1,1),
    decomposition=[
        (0,0,init_theta)
    ],
    target=(0,1),
    initial_parameter=init_theta,
    learning_rate=2.0,
    max_iter=60,
    shots=200000,
    delta=4e-3
)

np.save(
    "cost_fd_qpu.npy",
    cost_fd
)

In [ ]:
probs_psr_all = []
probs_fd_all = []

for index in range(5):
    probs_psr = np.load(f"cost_psr_qpu_{index}.npy")
    probs_fd = np.load(f"cost_fd_qpu_{index}.npy")

    probs_psr_all.append(probs_psr)
    probs_fd_all.append(probs_fd)

probs_psr_all = np.array(probs_psr_all)
probs_fd_all = np.array(probs_fd_all)

psr_mean = np.mean(probs_psr_all, axis=0)
psr_std = np.std(probs_psr_all, axis=0)

fd_mean = np.mean(probs_fd_all, axis=0)
fd_std = np.std(probs_fd_all, axis=0)

iterations = np.arange(len(psr_mean))

plt.figure(figsize=(7, 5))

plt.plot(
    iterations,
    psr_mean,
    "o-",
    label="PSR"
)

plt.fill_between(
    iterations,
    psr_mean - psr_std,
    psr_mean + psr_std,
    alpha=0.2
)

plt.plot(
    iterations,
    fd_mean,
    "s-",
    label="FD"
)

plt.fill_between(
    iterations,
    np.maximum(fd_mean - fd_std, 0),
    fd_mean + fd_std,
    alpha=0.2
)

plt.xlabel("Iteration")
plt.ylabel("Probability")

plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    "probability_vs_iteration.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.show()